<a href="https://colab.research.google.com/github/FlavioMartelli/Pokemon-Battle/blob/Giulia/prova_fds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

import os
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)



In [10]:
import os
import json
import pandas as pd
import numpy as np

if os.path.exists('/content/train.jsonl'):
    base_path = '/content/'
elif os.path.exists('/kaggle/input/competitions/fds-pokemon-battles-prediction-2026/'):
    base_path = '/kaggle/input/competitions/fds-pokemon-battles-prediction-2026/'
else:
    base_path = './'

print(f"Percorso utilizzato per i dati: {base_path}")
type_df = pd.read_csv(os.path.join(base_path, 'type_chart_gen2.csv'))

# Tipi Fisici vs Speciali in Gen 2 (Punto 8)
PHYSICAL_TYPES = {'normal', 'fighting', 'flying', 'poison', 'ground', 'rock', 'bug', 'ghost', 'steel'}
SPECIAL_TYPES = {'fire', 'water', 'grass', 'electric', 'psychic', 'ice', 'dragon', 'dark'}

def get_type_multiplier(atk_type, def_types):
    """Calcola il moltiplicatore d'efficacia tenendo conto di immunità e debolezze doppie (Punto 2)."""
    mult = 1.0
    for dt in def_types:
        if dt == 'notype':
            continue
        match = type_df[(type_df['attacking_type'] == atk_type) & (type_df['defending_type'] == dt)]
        if not match.empty:
            mult *= match.iloc[0]['multiplier']
    return mult

def extract_features(data):
    features_list = []

    for battle in data:
        feat = {}
        feat['battle_id'] = battle.get('battle_id')

        # -------------------------------------------------------------
        # 1. STATISTICHE, MATCHUP LEAD & SPEED TIES (Punti 1 & 8)
        # -------------------------------------------------------------
        p1_team = battle.get('p1_team_details', [])
        p1_lead = p1_team[0] if p1_team else {}
        p2_lead = battle.get('p2_lead_details', {})

        # Somma statistiche divise tra Fisico e Speciale (Punto 1 & 8)
        feat['p1_sum_hp'] = sum(p.get('base_hp', 0) for p in p1_team)
        feat['p1_sum_atk'] = sum(p.get('base_atk', 0) for p in p1_team)
        feat['p1_sum_def'] = sum(p.get('base_def', 0) for p in p1_team)
        feat['p1_sum_spa'] = sum(p.get('base_spa', 0) for p in p1_team)
        feat['p1_sum_spd'] = sum(p.get('base_spd', 0) for p in p1_team)
        feat['p1_sum_spe'] = sum(p.get('base_spe', 0) for p in p1_team)
        feat['p1_bst_total'] = feat['p1_sum_hp'] + feat['p1_sum_atk'] + feat['p1_sum_def'] + feat['p1_sum_spa'] + feat['p1_sum_spd'] + feat['p1_sum_spe']

        # Matchup Lead & Speed Tie / Priorità Velocità (Punto 1)
        feat['lead_spe_diff'] = p1_lead.get('base_spe', 0) - p2_lead.get('base_spe', 0)
        feat['lead_p1_faster'] = int(p1_lead.get('base_spe', 0) > p2_lead.get('base_spe', 0))
        feat['lead_atk_vs_def'] = p1_lead.get('base_atk', 0) - p2_lead.get('base_def', 0)
        feat['lead_spa_vs_spd'] = p1_lead.get('base_spa', 0) - p2_lead.get('base_spd', 0)

        # -------------------------------------------------------------
        # 2. TIPI, DEBOLEZZE MULTIPLE & STAB LEAD (Punto 2)
        # -------------------------------------------------------------
        p1_lead_types = [t for t in p1_lead.get('types', []) if t != 'notype']
        p2_lead_types = [t for t in p2_lead.get('types', []) if t != 'notype']

        p1_lead_mult = max([get_type_multiplier(t, p2_lead_types) for t in p1_lead_types] or [1.0])
        p2_lead_mult = max([get_type_multiplier(t, p1_lead_types) for t in p2_lead_types] or [1.0])

        feat['lead_type_adv_p1'] = p1_lead_mult
        feat['lead_type_adv_p2'] = p2_lead_mult
        feat['lead_type_ratio'] = p1_lead_mult / (p2_lead_mult + 1e-5)

        # Presenza Tipi Acciaio e Buio in P1 (Punto 2)
        p1_all_types = set(t for p in p1_team for t in p.get('types', []) if t != 'notype')
        feat['p1_has_steel'] = int('steel' in p1_all_types)
        feat['p1_has_dark'] = int('dark' in p1_all_types)

        # -------------------------------------------------------------
        # 3. STRUMENTI TENUTI - HELD ITEMS (Punto 3)
        # -------------------------------------------------------------
        items = [p.get('item', '') for p in p1_team]
        feat['count_leftovers'] = items.count('leftovers')
        feat['p1_lead_leftovers'] = int(p1_lead.get('item') == 'leftovers')
        feat['has_thickclub_marowak'] = int(any(p.get('name') == 'marowak' and p.get('item') == 'thickclub' for p in p1_team))
        feat['count_berserk_gene'] = items.count('berserkgene')
        feat['count_berry_items'] = sum(1 for item in items if item in ['mintberry', 'miracleberry', 'bitterberry', 'przcureberry'])
        feat['count_no_item'] = sum(1 for item in items if item in ['', 'noitem', None])

        # -------------------------------------------------------------
        # 4, 5, 6, 7. METRICHE DELLA TIMELINE (Punti 4, 5, 6, 7 & 8)
        # -------------------------------------------------------------
        timeline = battle.get('battle_timeline', [])
        feat['num_turns'] = len(timeline)

        if len(timeline) > 0:
            last_turn = timeline[-1]
            p1_end = last_turn['p1_pokemon_state']
            p2_end = last_turn['p2_pokemon_state']

            # HP finali e KO (Punto 4)
            feat['p1_final_hp'] = p1_end['hp_pct']
            feat['p2_final_hp'] = p2_end['hp_pct']
            feat['hp_diff'] = feat['p1_final_hp'] - feat['p2_final_hp']

            feat['p1_is_fainted'] = int(p1_end['status'] == 'fnt')
            feat['p2_is_fainted'] = int(p2_end['status'] == 'fnt')
            feat['p1_fainted_count'] = len(set(t['p1_pokemon_state']['name'] for t in timeline if t['p1_pokemon_state']['status'] == 'fnt'))
            feat['p2_fainted_count'] = len(set(t['p2_pokemon_state']['name'] for t in timeline if t['p2_pokemon_state']['status'] == 'fnt'))

            # Switch & Entry Hazards Spikes (Punto 4)
            feat['p1_switches'] = sum(1 for t in timeline if t['p1_move_details'] is None)
            feat['p2_switches'] = sum(1 for t in timeline if t['p2_move_details'] is None)

            # Stat Boosts & Phasing/Roar/Whirlwind (Punto 5)
            p1_boost_sum = sum(p1_end['boosts'].values())
            p2_boost_sum = sum(p2_end['boosts'].values())
            feat['p1_final_boosts'] = p1_boost_sum
            feat['p2_final_boosts'] = p2_boost_sum
            feat['boost_diff'] = p1_boost_sum - p2_boost_sum

            # Problemi di Stato Primari (Punto 6)
            feat['p1_sleep_turns'] = sum(1 for t in timeline if t['p1_pokemon_state']['status'] == 'slp')
            feat['p2_sleep_turns'] = sum(1 for t in timeline if t['p2_pokemon_state']['status'] == 'slp')
            feat['p1_freeze_turns'] = sum(1 for t in timeline if t['p1_pokemon_state']['status'] == 'frz')
            feat['p2_freeze_turns'] = sum(1 for t in timeline if t['p2_pokemon_state']['status'] == 'frz')
            feat['p1_paralysis_turns'] = sum(1 for t in timeline if t['p1_pokemon_state']['status'] == 'par')
            feat['p2_paralysis_turns'] = sum(1 for t in timeline if t['p2_pokemon_state']['status'] == 'par')
            feat['p1_burn_turns'] = sum(1 for t in timeline if t['p1_pokemon_state']['status'] == 'brn')
            feat['p2_burn_turns'] = sum(1 for t in timeline if t['p2_pokemon_state']['status'] == 'brn')
            feat['p1_toxic_turns'] = sum(1 for t in timeline if t['p1_pokemon_state']['status'] == 'tox')
            feat['p2_toxic_turns'] = sum(1 for t in timeline if t['p2_pokemon_state']['status'] == 'tox')

            # Mosse ad Alto Impatto (Punto 7)
            p1_moves = [t['p1_move_details'] for t in timeline if t.get('p1_move_details') is not None]
            p2_moves = [t['p2_move_details'] for t in timeline if t.get('p2_move_details') is not None]

            p1_move_names = [m.get('name', '').lower() for m in p1_moves]
            p2_move_names = [m.get('name', '').lower() for m in p2_moves]

            feat['p1_used_explosion'] = sum(1 for m in p1_move_names if m in ['explosion', 'self-destruct', 'selfdestruct'])
            feat['p2_used_explosion'] = sum(1 for m in p2_move_names if m in ['explosion', 'self-destruct', 'selfdestruct'])
            feat['p1_used_restalk'] = sum(1 for m in p1_move_names if m in ['rest', 'sleeptalk'])
            feat['p2_used_restalk'] = sum(1 for m in p2_move_names if m in ['rest', 'sleeptalk'])
            feat['p1_used_weather'] = sum(1 for m in p1_move_names if m in ['sandstorm', 'sunnyday', 'raindance'])
            feat['p2_used_weather'] = sum(1 for m in p2_move_names if m in ['sandstorm', 'sunnyday', 'raindance'])
            feat['p1_used_phazing'] = sum(1 for m in p1_move_names if m in ['roar', 'whirlwind'])
            feat['p2_used_phazing'] = sum(1 for m in p2_move_names if m in ['roar', 'whirlwind'])

            # Potenza media & Categoria Danno Fisico vs Speciale + STAB (Punto 2 & 8)
            feat['p1_avg_power'] = np.mean([m.get('base_power', 0) for m in p1_moves]) if p1_moves else 0
            feat['p2_avg_power'] = np.mean([m.get('base_power', 0) for m in p2_moves]) if p2_moves else 0

            feat['p1_physical_move_count'] = sum(1 for m in p1_moves if m.get('type') in PHYSICAL_TYPES)
            feat['p1_special_move_count'] = sum(1 for m in p1_moves if m.get('type') in SPECIAL_TYPES)
            feat['p2_physical_move_count'] = sum(1 for m in p2_moves if m.get('type') in PHYSICAL_TYPES)
            feat['p2_special_move_count'] = sum(1 for m in p2_moves if m.get('type') in SPECIAL_TYPES)

        else:
            # Valori di default per lotte a 0 turni
            for k in ['p1_final_hp', 'p2_final_hp', 'hp_diff', 'p1_is_fainted', 'p2_is_fainted',
                      'p1_fainted_count', 'p2_fainted_count', 'p1_switches', 'p2_switches',
                      'p1_final_boosts', 'p2_final_boosts', 'boost_diff', 'p1_sleep_turns',
                      'p2_sleep_turns', 'p1_freeze_turns', 'p2_freeze_turns', 'p1_paralysis_turns',
                      'p2_paralysis_turns', 'p1_burn_turns', 'p2_burn_turns', 'p1_toxic_turns',
                      'p2_toxic_turns', 'p1_used_explosion', 'p2_used_explosion', 'p1_used_restalk',
                      'p2_used_restalk', 'p1_used_weather', 'p2_used_weather', 'p1_used_phazing',
                      'p2_used_phazing', 'p1_avg_power', 'p2_avg_power', 'p1_physical_move_count',
                      'p1_special_move_count', 'p2_physical_move_count', 'p2_special_move_count']:
                feat[k] = 0
            feat['p1_final_hp'] = 1.0
            feat['p2_final_hp'] = 1.0

        if 'player_won' in battle:
            feat['player_won'] = int(battle['player_won'])

        features_list.append(feat)

    return pd.DataFrame(features_list).fillna(0)

Percorso utilizzato per i dati: /content/


In [ ]:
# -*- coding: utf-8 -*-
"""
Pokemon Gen 2 OU - modello V4
"""
import os
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError("XGBoost non è installato. Esegui: pip install xgboost") from exc

# ----------------------------
# 1. PERCORSI E CONFIGURAZIONE
# ----------------------------

if os.path.exists("/content/train.jsonl"):
    base_path = "/content/"
elif os.path.exists(
    "/kaggle/input/competitions/fds-pokemon-battles-prediction-2026/"
):
    base_path = (
        "/kaggle/input/competitions/"
        "fds-pokemon-battles-prediction-2026/"
    )
else:
    base_path = "./"

print("Percorso dati:", base_path)

type_df = pd.read_csv(
    os.path.join(base_path, "type_chart_gen2.csv")
)

# Normalizza i nomi dei tipi nella type chart.
type_df["attacking_type"] = (
    type_df["attacking_type"].astype(str).str.lower()
)
type_df["defending_type"] = (
    type_df["defending_type"].astype(str).str.lower()
)

TYPE_MULTIPLIER = {
    (row.attacking_type, row.defending_type): float(row.multiplier)
    for row in type_df.itertuples()
}

PHYSICAL_TYPES = {
    "normal", "fighting", "flying", "poison", "ground",
    "rock", "bug", "ghost", "steel"
}
SPECIAL_TYPES = {
    "fire", "water", "grass", "electric",
    "psychic", "ice", "dragon", "dark"
}

STAT_KEYS = [
    "base_hp", "base_atk", "base_def",
    "base_spa", "base_spd", "base_spe"
]

STAT_SHORT = ["hp", "atk", "def", "spa", "spd", "spe"]

STATUS_LIST = ["slp", "frz", "par", "brn", "tox"]
IMPORTANT_ITEMS = [
    "leftovers", "thickclub", "lightball", "berserkgene",
    "mintberry", "miracleberry", "bitterberry", "przcureberry",
    "quickclaw", "kingsrock", "focusband", "metalpowder",
    "stick"
]

STRATEGIC_MOVES = {
    "curse", "rest", "sleeptalk", "spikes", "rapidspin",
    "explosion", "selfdestruct", "self-destruct", "toxic",
    "thunderwave", "stunspore", "roar", "whirlwind",
    "reflect", "lightscreen", "light-screen", "protect",
    "substitute", "bellydrum", "swordsdance", "recover",
    "softboiled", "moonlight", "morningsun", "synthesis",
    "perishsong", "encore", "batonpass", "haze",
    "sandstorm", "raindance", "sunnyday"
}

# Indicatori strategici basati su specie comunemente rilevanti in GSC OU.
# Sono feature descrittive, non assunzioni sul moveset.
OU_SPECIES = {
    "snorlax", "zapdos", "raikou", "skarmory", "cloyster",
    "tyranitar", "marowak", "umbreon", "suicune", "exeggutor",
    "machamp", "nidoking", "heracross", "steelix", "starmie",
    "forretress", "blissey", "miltank", " gengar".strip()
}


# ----------------------------
# 2. FUNZIONI DI SUPPORTO
# ----------------------------

def safe_float(value, default=0.0):
    try:
        if value is None:
            return default
        value = float(value)
        if not np.isfinite(value):
            return default
        return value
    except (TypeError, ValueError):
        return default


def clean_type_list(types):
    if not isinstance(types, list):
        return []
    return [
        str(t).lower()
        for t in types
        if t is not None and str(t).lower() not in ("notype", "")
    ]


def get_type_multiplier(attack_type, defender_types):
    """Moltiplicatore Gen 2 per una mossa contro uno o due tipi."""
    attack_type = str(attack_type).lower()
    multiplier = 1.0

    for defender_type in clean_type_list(defender_types):
        multiplier *= TYPE_MULTIPLIER.get(
            (attack_type, defender_type), 1.0
        )

    return multiplier


def get_bst(pokemon):
    return sum(safe_float(pokemon.get(k, 0)) for k in STAT_KEYS)


def get_state(turn, player):
    state = turn.get(f"{player}_pokemon_state", {})
    return state if isinstance(state, dict) else {}


def get_move(turn, player):
    move = turn.get(f"{player}_move_details")
    return move if isinstance(move, dict) else None


def get_boost_sum(state):
    boosts = state.get("boosts", {})
    if not isinstance(boosts, dict):
        return 0.0
    return sum(safe_float(v) for v in boosts.values())


def hp_value(state):
    return safe_float(state.get("hp_pct", 0.0), 0.0)


def get_name(state):
    return str(state.get("name", "")).lower()


def count_unique_fainted(timeline, player):
    names = set()
    for turn in timeline:
        state = get_state(turn, player)
        if state.get("status") == "fnt" and get_name(state):
            names.add(get_name(state))
    return len(names)


def add_team_features(feat, team, prefix):
    """Statistiche, tipi, strumenti e ruoli della squadra conosciuta."""
    team = team if isinstance(team, list) else []

    feat[f"{prefix}_team_size"] = len(team)

    for stat_key, short in zip(STAT_KEYS, STAT_SHORT):
        values = [safe_float(p.get(stat_key, 0)) for p in team]
        feat[f"{prefix}_sum_{short}"] = sum(values)
        feat[f"{prefix}_mean_{short}"] = np.mean(values) if values else 0
        feat[f"{prefix}_max_{short}"] = max(values) if values else 0
        feat[f"{prefix}_min_{short}"] = min(values) if values else 0
        feat[f"{prefix}_std_{short}"] = np.std(values) if values else 0

    bsts = [get_bst(p) for p in team]
    feat[f"{prefix}_bst_sum"] = sum(bsts)
    feat[f"{prefix}_bst_mean"] = np.mean(bsts) if bsts else 0
    feat[f"{prefix}_bst_max"] = max(bsts) if bsts else 0

    # Profili statistici della squadra.
    atk = [safe_float(p.get("base_atk", 0)) for p in team]
    spa = [safe_float(p.get("base_spa", 0)) for p in team]
    deff = [safe_float(p.get("base_def", 0)) for p in team]
    spd = [safe_float(p.get("base_spd", 0)) for p in team]
    spe = [safe_float(p.get("base_spe", 0)) for p in team]
    hp = [safe_float(p.get("base_hp", 0)) for p in team]
    feat[f"{prefix}_physical_offense_mean"] = float(np.mean(atk)) if atk else 0
    feat[f"{prefix}_special_offense_mean"] = float(np.mean(spa)) if spa else 0
    feat[f"{prefix}_physical_bulk_mean"] = float(np.mean(deff)) if deff else 0
    feat[f"{prefix}_special_bulk_mean"] = float(np.mean(spd)) if spd else 0
    feat[f"{prefix}_speed_mean"] = float(np.mean(spe)) if spe else 0
    feat[f"{prefix}_speed_max"] = max(spe) if spe else 0
    feat[f"{prefix}_fast_count_100"] = sum(v >= 100 for v in spe)
    feat[f"{prefix}_fast_count_80"] = sum(v >= 80 for v in spe)
    feat[f"{prefix}_physical_bias"] = (
        float(np.mean(atk)) - float(np.mean(spa)) if atk and spa else 0
    )
    feat[f"{prefix}_bulk_mean"] = (
        (float(np.mean(hp)) + float(np.mean(deff)) + float(np.mean(spd))) / 3
        if hp and deff and spd else 0
    )

    # Tipi presenti nella squadra.
    all_types = []
    for pokemon in team:
        all_types.extend(clean_type_list(pokemon.get("types", [])))

    for pokemon_type in sorted(
        PHYSICAL_TYPES | SPECIAL_TYPES
    ):
        feat[f"{prefix}_type_count_{pokemon_type}"] = (
            all_types.count(pokemon_type)
        )

    feat[f"{prefix}_unique_types"] = len(set(all_types))
    feat[f"{prefix}_has_steel"] = int("steel" in all_types)
    feat[f"{prefix}_has_dark"] = int("dark" in all_types)
    feat[f"{prefix}_has_ghost"] = int("ghost" in all_types)
    feat[f"{prefix}_has_flying"] = int("flying" in all_types)

    # Strumenti.
    items = [
        str(p.get("item", "") or "").lower()
        for p in team
    ]
    for item in IMPORTANT_ITEMS:
        feat[f"{prefix}_item_{item}"] = items.count(item)

    feat[f"{prefix}_no_item_count"] = sum(
        item in ("", "noitem", "none") for item in items
    )

    # Pokémon e combinazioni note del metagame.
    names = [str(p.get("name", "")).lower() for p in team]
    for species in sorted(OU_SPECIES):
        feat[f"{prefix}_species_{species}"] = int(species in names)

    feat[f"{prefix}_has_marowak_thickclub"] = int(
        any(
            str(p.get("name", "")).lower() == "marowak"
            and str(p.get("item", "")).lower() == "thickclub"
            for p in team
        )
    )
    feat[f"{prefix}_has_pikachu_lightball"] = int(
        any(
            str(p.get("name", "")).lower() == "pikachu"
            and str(p.get("item", "")).lower() == "lightball"
            for p in team
        )
    )


# ----------------------------
# 3. FEATURE ENGINEERING
# ----------------------------

def extract_features(data):
    rows = []

    for battle in data:
        feat = {}
        feat["battle_id"] = battle.get("battle_id")

        p1_team = battle.get("p1_team_details", [])
        if not isinstance(p1_team, list):
            p1_team = []

        p1_lead = p1_team[0] if p1_team else {}
        p2_lead = battle.get("p2_lead_details", {})
        if not isinstance(p2_lead, dict):
            p2_lead = {}

        # ----- Squadre e statistiche base -----
        add_team_features(feat, p1_team, "p1")

        # Per P2 conosciamo soltanto il lead: non inventiamo gli altri 5.
        add_team_features(feat, [p2_lead] if p2_lead else [], "p2lead")

        # Statistiche e differenze del lead.
        for stat_key, short in zip(STAT_KEYS, STAT_SHORT):
            p1_value = safe_float(p1_lead.get(stat_key, 0))
            p2_value = safe_float(p2_lead.get(stat_key, 0))
            feat[f"lead_p1_{short}"] = p1_value
            feat[f"lead_p2_{short}"] = p2_value
            feat[f"lead_diff_{short}"] = p1_value - p2_value

        feat["lead_p1_bst"] = get_bst(p1_lead)
        feat["lead_p2_bst"] = get_bst(p2_lead)
        feat["lead_bst_diff"] = (
            feat["lead_p1_bst"] - feat["lead_p2_bst"]
        )
        feat["lead_p1_faster"] = int(
            safe_float(p1_lead.get("base_spe", 0))
            > safe_float(p2_lead.get("base_spe", 0))
        )
        for stat in ["base_atk", "base_def", "base_spa", "base_spd", "base_hp", "base_spe"]:
            a = safe_float(p1_lead.get(stat, 0))
            b = safe_float(p2_lead.get(stat, 0))
            feat[f"lead_ratio_{stat}"] = a / (b + 1.0)
        feat["lead_p1_physical_bias"] = (
            safe_float(p1_lead.get("base_atk", 0)) -
            safe_float(p1_lead.get("base_spa", 0))
        )
        feat["lead_p2_physical_bias"] = (
            safe_float(p2_lead.get("base_atk", 0)) -
            safe_float(p2_lead.get("base_spa", 0))
        )

        # Matchup difensivo: quanto i tipi del lead resistono agli attacchi
        # dei tipi avversari. È una proxy basata sui tipi, non sul moveset.
        p1_types = clean_type_list(p1_lead.get("types", []))
        p2_types = clean_type_list(p2_lead.get("types", []))

        for attack_type in sorted(PHYSICAL_TYPES | SPECIAL_TYPES):
            feat[f"p1lead_def_mult_from_{attack_type}"] = (
                get_type_multiplier(attack_type, p1_types)
            )
            feat[f"p2lead_def_mult_from_{attack_type}"] = (
                get_type_multiplier(attack_type, p2_types)
            )

        p1_best_type_matchup = [
            get_type_multiplier(t, p2_types) for t in p1_types
        ]
        p2_best_type_matchup = [
            get_type_multiplier(t, p1_types) for t in p2_types
        ]
        feat["lead_best_type_attack_p1"] = (
            max(p1_best_type_matchup) if p1_best_type_matchup else 1.0
        )
        feat["lead_best_type_attack_p2"] = (
            max(p2_best_type_matchup) if p2_best_type_matchup else 1.0
        )

        # ----- Timeline -----
        timeline = battle.get("battle_timeline", [])
        if not isinstance(timeline, list):
            timeline = []

        feat["num_turns"] = len(timeline)

        p1_states = [get_state(t, "p1") for t in timeline]
        p2_states = [get_state(t, "p2") for t in timeline]

        p1_hp = [hp_value(s) for s in p1_states]
        p2_hp = [hp_value(s) for s in p2_states]
        hp_diffs = [a - b for a, b in zip(p1_hp, p2_hp)]

        # Snapshot HP a turni prefissati: ultimo stato disponibile <= turno.
        for checkpoint in [1, 3, 5, 10, 15, 20, 25, 30]:
            idx = min(checkpoint - 1, len(timeline) - 1)
            if idx >= 0:
                feat[f"hp_diff_turn_{checkpoint}"] = hp_diffs[idx]
                feat[f"p1_hp_turn_{checkpoint}"] = p1_hp[idx]
                feat[f"p2_hp_turn_{checkpoint}"] = p2_hp[idx]
            else:
                feat[f"hp_diff_turn_{checkpoint}"] = 0
                feat[f"p1_hp_turn_{checkpoint}"] = 0
                feat[f"p2_hp_turn_{checkpoint}"] = 0

        if timeline:
            feat["p1_last_active_hp"] = p1_hp[-1]
            feat["p2_last_active_hp"] = p2_hp[-1]
            feat["last_active_hp_diff"] = hp_diffs[-1]
            feat["p1_last_active_fainted"] = int(
                p1_states[-1].get("status") == "fnt"
            )
            feat["p2_last_active_fainted"] = int(
                p2_states[-1].get("status") == "fnt"
            )
        else:
            for key in [
                "p1_last_active_hp", "p2_last_active_hp",
                "last_active_hp_diff", "p1_last_active_fainted",
                "p2_last_active_fainted"
            ]:
                feat[key] = 0

        # HP trend: regressione lineare sul differenziale osservato.
        if len(hp_diffs) >= 2:
            feat["hp_diff_trend"] = float(
                np.polyfit(np.arange(len(hp_diffs)), hp_diffs, 1)[0]
            )
            feat["hp_diff_mean"] = float(np.mean(hp_diffs))
            feat["hp_diff_min"] = float(np.min(hp_diffs))
            feat["hp_diff_max"] = float(np.max(hp_diffs))
            feat["hp_diff_std"] = float(np.std(hp_diffs))
        else:
            feat["hp_diff_trend"] = 0
            feat["hp_diff_mean"] = 0
            feat["hp_diff_min"] = 0
            feat["hp_diff_max"] = 0
            feat["hp_diff_std"] = 0

        # Variazione HP dell'attivo tra primo e ultimo snapshot.
        feat["p1_active_hp_change"] = (
            p1_hp[-1] - p1_hp[0] if len(p1_hp) >= 2 else 0
        )
        feat["p2_active_hp_change"] = (
            p2_hp[-1] - p2_hp[0] if len(p2_hp) >= 2 else 0
        )

        # Fasi della lotta.
        for phase, start, end in [
            ("early", 0, 5),
            ("mid", 5, 15),
            ("late", 15, 30)
        ]:
            diffs = hp_diffs[start:end]
            feat[f"{phase}_hp_diff_mean"] = (
                float(np.mean(diffs)) if diffs else 0
            )
            feat[f"{phase}_hp_diff_last"] = (
                float(diffs[-1]) if diffs else 0
            )

        # KO distinti osservati nella timeline (solo Pokémon effettivamente visti).
        feat["p1_observed_fainted_count"] = count_unique_fainted(
            timeline, "p1"
        )
        feat["p2_observed_fainted_count"] = count_unique_fainted(
            timeline, "p2"
        )
        feat["observed_fainted_diff"] = (
            feat["p2_observed_fainted_count"]
            - feat["p1_observed_fainted_count"]
        )

        # Pokémon attivi distinti e cambi di specie osservati.
        p1_names = [get_name(s) for s in p1_states if get_name(s)]
        p2_names = [get_name(s) for s in p2_states if get_name(s)]

        feat["p1_distinct_active_pokemon"] = len(set(p1_names))
        feat["p2_distinct_active_pokemon"] = len(set(p2_names))

        feat["p1_switches_observed"] = sum(
            a != b for a, b in zip(p1_names[:-1], p1_names[1:])
        )
        feat["p2_switches_observed"] = sum(
            a != b for a, b in zip(p2_names[:-1], p2_names[1:])
        )

        # Status: turn-count e specie distinte colpite.
        for player, states in [("p1", p1_states), ("p2", p2_states)]:
            for status in STATUS_LIST:
                status_states = [
                    s for s in states if s.get("status") == status
                ]
                feat[f"{player}_{status}_turns"] = len(status_states)
                feat[f"{player}_{status}_pokemon_count"] = len({
                    get_name(s) for s in status_states if get_name(s)
                })

            feat[f"{player}_boost_sum_mean"] = (
                np.mean([get_boost_sum(s) for s in states])
                if states else 0
            )
            feat[f"{player}_boost_sum_max"] = (
                max([get_boost_sum(s) for s in states], default=0)
            )
            feat[f"{player}_boost_sum_last"] = (
                get_boost_sum(states[-1]) if states else 0
            )

        feat["boost_last_diff"] = (
            feat["p1_boost_sum_last"] - feat["p2_boost_sum_last"]
        )

        # Mosse osservate.
        p1_moves = [get_move(t, "p1") for t in timeline]
        p2_moves = [get_move(t, "p2") for t in timeline]
        p1_moves = [m for m in p1_moves if m is not None]
        p2_moves = [m for m in p2_moves if m is not None]

        for player, moves in [("p1", p1_moves), ("p2", p2_moves)]:
            names = [
                str(m.get("name", "")).lower().replace(" ", "")
                for m in moves
            ]
            powers = [
                safe_float(m.get("base_power", 0))
                for m in moves
            ]

            feat[f"{player}_move_count"] = len(moves)
            feat[f"{player}_avg_move_power"] = (
                float(np.mean(powers)) if powers else 0
            )
            feat[f"{player}_max_move_power"] = (
                max(powers) if powers else 0
            )
            feat[f"{player}_high_power_moves"] = sum(
                power >= 100 for power in powers
            )

            for move_name in sorted(STRATEGIC_MOVES):
                normalized = move_name.replace("-", "").replace(" ", "")
                feat[f"{player}_move_{normalized}"] = sum(
                    n == normalized for n in names
                )

            feat[f"{player}_explosion_count"] = sum(
                n in ("explosion", "selfdestruct") for n in names
            )
            feat[f"{player}_rest_count"] = names.count("rest")
            feat[f"{player}_sleeptalk_count"] = names.count("sleeptalk")
            feat[f"{player}_spikes_count"] = names.count("spikes")
            feat[f"{player}_rapidspin_count"] = names.count("rapidspin")
            feat[f"{player}_curse_count"] = names.count("curse")
            feat[f"{player}_toxic_count"] = names.count("toxic")
            feat[f"{player}_thunderwave_count"] = names.count("thunderwave")

            # Categorie dichiarate nel dataset; il fallback per tipo
            # segue la separazione fisico/speciale della Gen 2.
            physical = 0
            special = 0
            status_moves = 0
            for move in moves:
                category = str(move.get("category", "")).lower()
                move_type = str(move.get("type", "")).lower()
                power = safe_float(move.get("base_power", 0))

                # Gen 2: fisico/speciale dipende dal TIPO, non dalla categoria moderna.
                if category in ("status", "non-damaging") or power <= 0:
                    status_moves += 1
                elif move_type in PHYSICAL_TYPES:
                    physical += 1
                elif move_type in SPECIAL_TYPES:
                    special += 1

            feat[f"{player}_physical_moves"] = physical
            feat[f"{player}_special_moves"] = special
            feat[f"{player}_status_moves"] = status_moves

        # RestTalk effettivamente osservato nella stessa timeline.
        feat["p1_resttalk_combo"] = int(
            feat["p1_rest_count"] > 0
            and feat["p1_sleeptalk_count"] > 0
        )
        feat["p2_resttalk_combo"] = int(
            feat["p2_rest_count"] > 0
            and feat["p2_sleeptalk_count"] > 0
        )

        # Copertura offensiva osservata: migliori efficacie delle mosse
        # del lead contro il lead avversario, limitata ai tipi disponibili.
        p1_lead_move_types = {
            str(m.get("type", "")).lower()
            for m in p1_moves
            if str(m.get("type", "")).lower() in (PHYSICAL_TYPES | SPECIAL_TYPES)
        }
        p2_lead_move_types = {
            str(m.get("type", "")).lower()
            for m in p2_moves
            if str(m.get("type", "")).lower() in (PHYSICAL_TYPES | SPECIAL_TYPES)
        }

        feat["p1_move_type_coverage_vs_p2lead"] = max(
            [
                get_type_multiplier(t, p2_types)
                for t in p1_lead_move_types
            ] or [1.0]
        )
        feat["p2_move_type_coverage_vs_p1lead"] = max(
            [
                get_type_multiplier(t, p1_types)
                for t in p2_lead_move_types
            ] or [1.0]
        )

        # Feature di archetipi dedotte dalle mosse realmente osservate.
        p1_move_set = {
            str(m.get("name", "")).lower().replace(" ", "")
            for m in p1_moves
        }
        p2_move_set = {
            str(m.get("name", "")).lower().replace(" ", "")
            for m in p2_moves
        }

        for player, move_set in [
            ("p1", p1_move_set), ("p2", p2_move_set)
        ]:
            feat[f"{player}_observed_curse_rest"] = int(
                "curse" in move_set and "rest" in move_set
            )
            feat[f"{player}_observed_spikes_spin"] = int(
                "spikes" in move_set and "rapidspin" in move_set
            )
            feat[f"{player}_observed_screen"] = int(
                "reflect" in move_set
                or "lightscreen" in move_set
                or "light-screen" in move_set
            )

        # Etichetta soltanto nel train.
        if "player_won" in battle:
            feat["player_won"] = int(bool(battle["player_won"]))

        rows.append(feat)

    return pd.DataFrame(rows).replace(
        [np.inf, -np.inf], np.nan
    ).fillna(0)


# ----------------------------
# 4. CARICAMENTO DATI
# ----------------------------

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                records.append(json.loads(line))
    return records


print("\nCaricamento train...")
train_data = load_jsonl(os.path.join(base_path, "train.jsonl"))

print("Caricamento test...")
test_data = load_jsonl(os.path.join(base_path, "test.jsonl"))

print("Estrazione feature train...")
train_df = extract_features(train_data)

print("Estrazione feature test...")
test_df = extract_features(test_data)

del train_data, test_data

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


# ----------------------------
# 5. MATRICE FEATURE
# ----------------------------

target = "player_won"
id_col = "battle_id"

features = [
    col for col in train_df.columns
    if col not in [target, id_col]
]

# Garantisce identiche colonne e ordine in train/test.
X_train = train_df[features].copy()
y_train = train_df[target].astype(int).copy()
X_test = test_df.reindex(columns=features, fill_value=0).copy()

# Tutte le colonne sono numeriche; imputazione dentro ogni pipeline
# per evitare che la statistica di imputazione guardi il validation fold.
print("Numero feature:", len(features))
print("Distribuzione target:")
print(y_train.value_counts(normalize=True))


# ----------------------------
# 6. MODELLI E CONFIGURAZIONI V4
# ----------------------------

from itertools import product

N_SPLITS = 5
RANDOM_STATE = 42

# I tre modelli di riferimento mantengono la configurazione V3.
base_models = {
    "Logistic Regression": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(
            C=0.5, max_iter=3000, solver="lbfgs",
            random_state=RANDOM_STATE
        )
    ),
    "Random Forest": make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=600, max_depth=None,
            min_samples_leaf=3, max_features="sqrt",
            n_jobs=-1, random_state=RANDOM_STATE,
            class_weight=None
        )
    ),
    "Extra Trees": make_pipeline(
        SimpleImputer(strategy="median"),
        ExtraTreesClassifier(
            n_estimators=600, max_depth=None,
            min_samples_leaf=3, max_features="sqrt",
            n_jobs=-1, random_state=RANDOM_STATE,
            class_weight=None
        )
    )
}

# Piccola ricerca manuale: mantiene il numero di fit ragionevole in Colab.
hist_configs = [
    {"max_iter": 400, "learning_rate": 0.03, "max_leaf_nodes": 15,
     "l2_regularization": 2.0},
    {"max_iter": 500, "learning_rate": 0.03, "max_leaf_nodes": 31,
     "l2_regularization": 3.0},
    {"max_iter": 650, "learning_rate": 0.025, "max_leaf_nodes": 15,
     "l2_regularization": 5.0},
]

xgb_configs = [
    {"n_estimators": 700, "max_depth": 3, "learning_rate": 0.03,
     "subsample": 0.85, "colsample_bytree": 0.85,
     "min_child_weight": 5, "reg_lambda": 8, "reg_alpha": 0.2,
     "gamma": 0.05},
    {"n_estimators": 900, "max_depth": 4, "learning_rate": 0.025,
     "subsample": 0.90, "colsample_bytree": 0.90,
     "min_child_weight": 4, "reg_lambda": 7, "reg_alpha": 0.15,
     "gamma": 0.05},
    {"n_estimators": 700, "max_depth": 5, "learning_rate": 0.02,
     "subsample": 0.85, "colsample_bytree": 0.85,
     "min_child_weight": 6, "reg_lambda": 10, "reg_alpha": 0.3,
     "gamma": 0.10},
]

def make_hist_model(params):
    return make_pipeline(
        SimpleImputer(strategy="median"),
        HistGradientBoostingClassifier(
            **params, random_state=RANDOM_STATE
        )
    )

def make_xgb_model(params):
    return make_pipeline(
        SimpleImputer(strategy="median"),
        XGBClassifier(
            **params,
            eval_metric="logloss",
            objective="binary:logistic",
            n_jobs=-1,
            random_state=RANDOM_STATE
        )
    )

skf = StratifiedKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE
)
folds = list(skf.split(X_train, y_train))

results = {}
oof_predictions = {}
test_predictions = {}

def evaluate_model(model_name, model_factory):
    """Calcola OOF e media delle predizioni test sui fold."""
    oof = np.zeros(len(X_train), dtype=float)
    test_pred = np.zeros(len(X_test), dtype=float)

    for fold, (train_idx, valid_idx) in enumerate(folds, start=1):
        X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[valid_idx]

        model = model_factory()
        model.fit(X_tr, y_tr)
        valid_prob = model.predict_proba(X_va)[:, 1]
        oof[valid_idx] = valid_prob
        test_pred += model.predict_proba(X_test)[:, 1] / N_SPLITS

        fold_acc = accuracy_score(y_va, (valid_prob >= 0.5).astype(int))
        print(f"  Fold {fold}: accuracy = {fold_acc:.5f}")

    score = accuracy_score(y_train, (oof >= 0.5).astype(int))
    print(f"  OOF accuracy {model_name}: {score:.5f}")
    return score, oof, test_pred

# Modelli di riferimento.
for name, prototype in base_models.items():
    print("\n" + "=" * 60 + "\nMODELLO:", name)
    score, oof, test_pred = evaluate_model(
        name, lambda p=prototype: p
    )
    results[name] = score
    oof_predictions[name] = oof
    test_predictions[name] = test_pred

# Selezione della configurazione HistGradientBoosting.
print("\n" + "=" * 60 + "\nRICERCA HISTGRADIENTBOOSTING")
hist_trials = []
for idx, params in enumerate(hist_configs, start=1):
    name = f"HistGradientBoosting_cfg{idx}"
    print(f"\nConfigurazione {idx}: {params}")
    score, oof, test_pred = evaluate_model(
        name, lambda p=params: make_hist_model(p)
    )
    hist_trials.append((score, oof, test_pred, params))

hist_score, hist_oof, hist_test, hist_best_params = max(
    hist_trials, key=lambda item: item[0]
)
results["HistGradientBoosting"] = hist_score
oof_predictions["HistGradientBoosting"] = hist_oof
test_predictions["HistGradientBoosting"] = hist_test
print("Migliori parametri HistGradientBoosting:", hist_best_params)

# Selezione della configurazione XGBoost.
print("\n" + "=" * 60 + "\nRICERCA XGBOOST")
xgb_trials = []
for idx, params in enumerate(xgb_configs, start=1):
    name = f"XGBoost_cfg{idx}"
    print(f"\nConfigurazione {idx}: {params}")
    score, oof, test_pred = evaluate_model(
        name, lambda p=params: make_xgb_model(p)
    )
    xgb_trials.append((score, oof, test_pred, params))

xgb_score, xgb_oof, xgb_test, xgb_best_params = max(
    xgb_trials, key=lambda item: item[0]
)
results["XGBoost"] = xgb_score
oof_predictions["XGBoost"] = xgb_oof
test_predictions["XGBoost"] = xgb_test
print("Migliori parametri XGBoost:", xgb_best_params)

# ----------------------------
# 7. CLASSIFICA DEI CINQUE MODELLI
# ----------------------------

print("\n\n========== MODEL RANKING V4 ==========")
ranking = sorted(results.items(), key=lambda item: item[1], reverse=True)
for position, (name, score) in enumerate(ranking, start=1):
    print(f"{position}. {name}: {score:.5f}")

# ----------------------------
# 8. ENSEMBLE PESATO DEI TRE MIGLIORI
# ----------------------------

best_names = [name for name, _ in ranking[:3]]
print("\nModelli candidati all'ensemble:", best_names)

# Ricerca su griglia di pesi (passi 0.05); i pesi sommano a 1.
# Si ottimizza anche la soglia, perché la metrica è accuracy.
best_combo_acc = -1.0
best_weights = None
best_threshold = 0.5
best_combo_oof = None

weight_values = np.arange(0.0, 1.0001, 0.05)
for w1 in weight_values:
    for w2 in weight_values:
        w3 = 1.0 - w1 - w2
        if w3 < -1e-9 or w3 > 1.0 + 1e-9:
            continue
        weights = np.array([w1, w2, max(0.0, w3)])
        if not np.isclose(weights.sum(), 1.0):
            continue

        combo_oof = sum(
            weights[i] * oof_predictions[name]
            for i, name in enumerate(best_names)
        )
        for threshold in np.arange(0.45, 0.551, 0.01):
            score = accuracy_score(
                y_train, (combo_oof >= threshold).astype(int)
            )
            if score > best_combo_acc:
                best_combo_acc = score
                best_weights = weights.copy()
                best_threshold = float(threshold)
                best_combo_oof = combo_oof.copy()

print(f"Ensemble pesato OOF accuracy: {best_combo_acc:.5f}")
print("Pesi:", dict(zip(best_names, best_weights.round(3))))
print(f"Soglia selezionata: {best_threshold:.2f}")

weighted_test = sum(
    best_weights[i] * test_predictions[name]
    for i, name in enumerate(best_names)
)

best_single_name = ranking[0][0]
best_single_acc = results[best_single_name]

print(f"Miglior modello singolo: {best_single_name} ({best_single_acc:.5f})")
print("ATTENZIONE: configurazioni, pesi e soglia sono selezionati sugli stessi")
print("OOF; il punteggio dell'ensemble è quindi una stima di selezione")
print("potenzialmente ottimistica, non una valutazione indipendente.")

if best_combo_acc >= best_single_acc:
    final_test_predictions = weighted_test
    final_threshold = best_threshold
    print("Submission basata sull'ensemble pesato.")
else:
    final_test_predictions = test_predictions[best_single_name]
    final_threshold = 0.5
    print("Submission basata sul miglior modello singolo.")

# ----------------------------
# 9. SUBMISSION
# ----------------------------

sample_sub = pd.read_csv(
    os.path.join(base_path, "sample_submission.csv")
)

preds_series = pd.Series(
    (final_test_predictions >= final_threshold).astype(int),
    index=test_df[id_col]
)

submission = sample_sub[[id_col]].copy()
submission[target] = submission[id_col].map(preds_series).astype(int)

assert list(submission.columns) == ["battle_id", "player_won"]
assert len(submission) == len(test_df)
assert submission["player_won"].notna().all()
assert set(submission["player_won"].unique()).issubset({0, 1})

submission.to_csv("submission.csv", index=False)
print("\nSubmission creata: submission.csv")
print(submission.head())
print("Dimensioni:", submission.shape)
